In [46]:
import pandas as pd
import numpy as np
import re
import scipy.sparse as sp
import pickle
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
print("All libraries imported successfully!")

All libraries imported successfully!


In [47]:
# Load raw datasets
df1 = pd.read_csv('spam.csv', encoding='latin-1')
df2 = pd.read_csv('SMS Spam Dataset.csv', encoding='latin-1')
df_synthetic = pd.read_csv('synthetic_scam_data.csv')

# Clean Dataset 1
df1 = df1[['v1', 'v2']]
df1.columns = ['label', 'message']
df1['label'] = df1['label'].map({'ham': 0, 'spam': 1})

# Clean Dataset 2
df2.columns = ['message', 'label']
df2 = df2[['label', 'message']]

# Clean synthetic data
df_synthetic = df_synthetic[['message', 'label']]
df_synthetic['label'] = df_synthetic['label'].astype(int)
df_synthetic = df_synthetic[['label', 'message']]

# Combine all three
df = pd.concat([df1, df2, df_synthetic], ignore_index=True)
df = df.drop_duplicates()
df = df.dropna()

print("Combined Dataset shape:", df.shape)
print("Label distribution:")
print(df['label'].value_counts())

Combined Dataset shape: (11407, 2)
Label distribution:
0    9122
1    2285
Name: label, dtype: int64


In [48]:
def count_urgency_words(text):
    urgency_words = ['urgent', 'immediately', 'now', 'hurry', 'quick',
                     'fast', 'limited', 'expire', 'deadline', 'asap',
                     'don\'t wait', 'act now', 'before it\'s too late']
    text = text.lower()
    return sum(1 for word in urgency_words if word in text)

def count_pressure_phrases(text):
    pressure_phrases = ['trust me', 'don\'t worry', 'i also did it',
                        'just ignore', 'it\'s fine', 'believe me',
                        'i promise', 'guaranteed', 'no risk']
    text = text.lower()
    return sum(1 for phrase in pressure_phrases if phrase in text)

def count_money_requests(text):
    money_words = ['send money', 'transfer', 'bank account', 'payment',
                   'crypto', 'bitcoin', 'gift card', 'wire', 'cash',
                   'invest', 'profit', 'earn', 'winning', 'prize']
    text = text.lower()
    return sum(1 for word in money_words if word in text)

def count_suspicious_links(text):
    url_pattern = re.compile(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+])+')
    return len(url_pattern.findall(text))

def count_secrecy_phrases(text):
    secrecy_words = ['don\'t tell', 'keep secret', 'between us',
                     'don\'t share', 'confidential', 'private',
                     'don\'t show', 'just you and me']
    text = text.lower()
    return sum(1 for phrase in secrecy_words if phrase in text)

print("Feature functions defined!")

Feature functions defined!


In [49]:
df['urgency_count'] = df['message'].apply(count_urgency_words)
df['pressure_count'] = df['message'].apply(count_pressure_phrases)
df['money_request_count'] = df['message'].apply(count_money_requests)
df['suspicious_links'] = df['message'].apply(count_suspicious_links)
df['secrecy_count'] = df['message'].apply(count_secrecy_phrases)
df['message_length'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda x: len(x.split()))

print("Features added!")
print("Dataset shape:", df.shape)
print("\nFeature summary for scam messages:")
print(df[df['label']==1][['urgency_count','pressure_count',
                           'money_request_count','suspicious_links',
                           'secrecy_count']].describe())

Features added!
Dataset shape: (11407, 9)

Feature summary for scam messages:
       urgency_count  pressure_count  money_request_count  suspicious_links  \
count    2285.000000     2285.000000          2285.000000       2285.000000   
mean        0.736980        0.094530             0.418381          0.129103   
std         0.855835        0.292628             0.655045          0.355661   
min         0.000000        0.000000             0.000000          0.000000   
25%         0.000000        0.000000             0.000000          0.000000   
50%         1.000000        0.000000             0.000000          0.000000   
75%         1.000000        0.000000             1.000000          0.000000   
max         4.000000        1.000000             3.000000          2.000000   

       secrecy_count  
count    2285.000000  
mean        0.119475  
std         0.332416  
min         0.000000  
25%         0.000000  
50%         0.000000  
75%         0.000000  
max         2.000000  


In [50]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_tfidf = tfidf.fit_transform(df['message'])

custom_features = df[['urgency_count', 'pressure_count',
                       'money_request_count', 'suspicious_links',
                       'secrecy_count', 'message_length',
                       'word_count']].values

X = sp.hstack([X_tfidf, custom_features])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Data preparation complete!")
print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")
print(f"Scam messages in training: {sum(y_train==1)}")
print(f"Legitimate messages in training: {sum(y_train==0)}")

Data preparation complete!
Training set size: (9125, 5007)
Testing set size: (2282, 5007)
Scam messages in training: 1828
Legitimate messages in training: 7297


In [51]:
model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Model trained successfully!")
print("\nClassification Report:")
print(classification_report(y_test, y_pred,
      target_names=['Legitimate', 'Scam']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Model trained successfully!

Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.99      0.99      0.99      1825
        Scam       0.94      0.97      0.96       457

    accuracy                           0.98      2282
   macro avg       0.97      0.98      0.97      2282
weighted avg       0.98      0.98      0.98      2282


Confusion Matrix:
[[1799   26]
 [  13  444]]


In [52]:
def analyse_message(message):
    urgency = count_urgency_words(message)
    pressure = count_pressure_phrases(message)
    money = count_money_requests(message)
    links = count_suspicious_links(message)
    secrecy = count_secrecy_phrases(message)
    length = len(message)
    words = len(message.split())

    message_tfidf = tfidf.transform([message])
    custom = [[urgency, pressure, money, links, secrecy, length, words]]
    message_features = sp.hstack([message_tfidf, custom])

    prediction = model.predict(message_features)[0]
    probability = model.predict_proba(message_features)[0][1]
    risk_score = int(probability * 100)

    if risk_score >= 70:
        risk_level = "High"
    elif risk_score >= 40:
        risk_level = "Medium"
    else:
        risk_level = "Low"

    triggered_features = []
    if urgency > 0: triggered_features.append("urgency")
    if pressure > 0: triggered_features.append("pressure_tactics")
    if money > 0: triggered_features.append("money_request")
    if links > 0: triggered_features.append("suspicious_links")
    if secrecy > 0: triggered_features.append("secrecy_tactics")
    if not triggered_features and prediction == 1:
        triggered_features.append("suspicious_language_pattern")

    escalate_to_analyst = (risk_score >= 90) and (secrecy > 0)

    return {
        "risk_score": risk_score,
        "risk_level": risk_level,
        "confidence": round(float(probability), 2),
        "triggered_features": triggered_features,
        "requires_human_review": risk_level in ["Medium", "High"],
        "escalate_to_analyst": escalate_to_analyst
    }

print("analyse_message function ready!")


analyse_message function ready!


In [54]:
print("Test 1 — Scam message:")
print(analyse_message("URGENT! Send money now to my account. Trust me don't worry. Limited time offer!"))

print("\nTest 2 — Legitimate message:")
print(analyse_message("Hey, are we still meeting for lunch tomorrow?"))

print("\nTest 3 — Social media impersonation:")
print(analyse_message("Hi it's me, I need you to transfer money urgently to this account, don't tell anyone please"))

print("\nTest 4 — Crypto scam:")
print(analyse_message("I made £500 in one day on this platform, you only need to invest £200 to start, DM me for the link"))

print("\nTest 5 — Romance scam:")
print(analyse_message("My darling I need your help, I am stuck at customs and need £300 for fees, please send urgently, don't tell anyone"))

Test 1 — Scam message:
{'risk_score': 99, 'risk_level': 'High', 'confidence': 1.0, 'triggered_features': ['urgency', 'pressure_tactics', 'money_request'], 'requires_human_review': True, 'escalate_to_analyst': False}

Test 2 — Legitimate message:
{'risk_score': 1, 'risk_level': 'Low', 'confidence': 0.01, 'triggered_features': [], 'requires_human_review': False, 'escalate_to_analyst': False}

Test 3 — Social media impersonation:
{'risk_score': 97, 'risk_level': 'High', 'confidence': 0.98, 'triggered_features': ['urgency', 'money_request', 'secrecy_tactics'], 'requires_human_review': True, 'escalate_to_analyst': True}

Test 4 — Crypto scam:
{'risk_score': 78, 'risk_level': 'High', 'confidence': 0.79, 'triggered_features': ['money_request'], 'requires_human_review': True, 'escalate_to_analyst': False}

Test 5 — Romance scam:
{'risk_score': 99, 'risk_level': 'High', 'confidence': 0.99, 'triggered_features': ['urgency', 'secrecy_tactics'], 'requires_human_review': True, 'escalate_to_analyst'

In [57]:
with open('scamshield_model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("Model saved as scamshield_model.pkl")
print("TF-IDF vectorizer saved as tfidf_vectorizer.pkl")

Model saved as scamshield_model.pkl
TF-IDF vectorizer saved as tfidf_vectorizer.pkl


In [55]:
print("Test 1 — Scam message:")
print(analyse_message("URGENT! Send money now to my account. Trust me don't worry. Limited time offer!"))

print("\nTest 2 — Legitimate message:")
print(analyse_message("Hey, are we still meeting for lunch tomorrow?"))

print("\nTest 3 — Social media impersonation:")
print(analyse_message("Hi it's me, I need you to transfer money urgently to this account, don't tell anyone please"))

print("\nTest 4 — Crypto scam:")
print(analyse_message("I made £500 in one day on this platform, you only need to invest £200 to start, DM me for the link"))

print("\nTest 5 — Romance scam:")
print(analyse_message("My darling I need your help, I am stuck at customs and need £300 for fees, please send urgently, don't tell anyone"))

Test 1 — Scam message:
{'risk_score': 99, 'risk_level': 'High', 'confidence': 1.0, 'triggered_features': ['urgency', 'pressure_tactics', 'money_request'], 'requires_human_review': True, 'escalate_to_analyst': False}

Test 2 — Legitimate message:
{'risk_score': 1, 'risk_level': 'Low', 'confidence': 0.01, 'triggered_features': [], 'requires_human_review': False, 'escalate_to_analyst': False}

Test 3 — Social media impersonation:
{'risk_score': 97, 'risk_level': 'High', 'confidence': 0.98, 'triggered_features': ['urgency', 'money_request', 'secrecy_tactics'], 'requires_human_review': True, 'escalate_to_analyst': True}

Test 4 — Crypto scam:
{'risk_score': 78, 'risk_level': 'High', 'confidence': 0.79, 'triggered_features': ['money_request'], 'requires_human_review': True, 'escalate_to_analyst': False}

Test 5 — Romance scam:
{'risk_score': 99, 'risk_level': 'High', 'confidence': 0.99, 'triggered_features': ['urgency', 'secrecy_tactics'], 'requires_human_review': True, 'escalate_to_analyst'